# Manual agent benchmark

Compare a **trained agent** loaded from a checkpoint against any subset of the heuristic baselines on a fixed evaluation scenario. Every agent is run on the **same factory layout** for `N_EVAL_EPISODES` rollouts of `MAX_EVAL_SIM_TIME` sim time each.

**What to edit:** only the *User configuration* cell below — the rest is wired up automatically.

**Outputs:**
- Per-episode KPI table (mean & std per agent)
- Bar charts comparing the headline KPIs
- Per-step trend plots (rolling-mean of `repair_time_delta`, `repair_quality`, fleet knowledge) averaged across episodes
- A CSV dump under `reports/benchmark_<timestamp>/`

## 1. User configuration

Point the three paths at the artefacts you want to evaluate, pick which baselines to compare against, and set the horizon. Paths are **relative to the project root** — the Setup cell (section 2) chdirs there before any file is read, so these stay correct regardless of where you launched Jupyter from.

In [ ]:
from pathlib import Path

# --- trained agent ---------------------------------------------------------
# Matches the 1000-episode PPO run:
#   kata-experiment \
#     --env run_configs/factory_long.json \
#     --agent run_configs/agents/ppo_transformer.json \
#     --experiment run_configs/experiments/train_long.json
TRAINED_ENV_CONFIG   = Path("run_configs/factory_long.json")            # env the agent was trained on
TRAINED_AGENT_CONFIG = Path("run_configs/agents/ppo_transformer.json")  # matching agent JSON
TRAINED_CHECKPOINT   = Path("checkpoints/ppo_transformer_best.pt")      # weights to load (save_best from train_long)
TRAINED_AGENT_LABEL  = "PPO-Transformer (1000 eps)"                     # label used in plots

# --- baselines to compare against ------------------------------------------
# Any subset of: 'random', 'round_robin', 'least_busy', 'least_fatigued', 'shortest_queue'
BASELINES: list[str] = [
    "random",
    "round_robin",
    "least_busy",
    "least_fatigued",
    "shortest_queue",
]

# --- evaluation horizon ----------------------------------------------------
# Defaults mirror factory_long.json's training horizon so the benchmark is
# directly comparable to the training-time eval curve.  Bump MAX_EVAL_SIM_TIME
# above 200_000 to probe extrapolation past the training horizon.
N_EVAL_EPISODES    = 5         # rollouts per agent (more = lower variance, longer wall time)
MAX_EVAL_SIM_TIME  = 200_000.0 # same as factory_long.json's max_sim_time
MAX_EVAL_STEPS     = 20_000    # same as factory_long.json's max_episode_steps
EVAL_SEED          = 4321      # seed for the fixed eval factory (RandomScenarioSampler)
DETERMINISTIC      = True      # use greedy policy for the trained agent

# --- output ----------------------------------------------------------------
REPORTS_ROOT = Path("reports")

print(f"Trained env config : {TRAINED_ENV_CONFIG}")
print(f"Trained agent cfg  : {TRAINED_AGENT_CONFIG}")
print(f"Checkpoint         : {TRAINED_CHECKPOINT}")
print(f"Baselines          : {BASELINES}")
print(f"Episodes / agent   : {N_EVAL_EPISODES}")
print(f"Max sim time       : {MAX_EVAL_SIM_TIME}")
print(f"Max env steps      : {MAX_EVAL_STEPS}")

## 2. Setup

In [ ]:
import json
import logging
import os
import sys
import time
import warnings
from contextlib import contextmanager

# Anchor to the repo root so the relative paths in the user-config cell
# resolve identically whether you launched Jupyter from the project
# root or from examples/.
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "run_configs").is_dir():
    _root = _root.parent
if (_root / "run_configs").is_dir():
    os.chdir(_root)
    print(f"Working directory: {_root}")
else:
    print(f"WARNING: could not locate repo root from {Path.cwd()}")

# Silence SimPy prints and excess library logging
logging.disable(logging.WARNING)
warnings.filterwarnings("ignore")

# Avoid loading an unrelated default config file
os.environ["KATA_CONF_PATH"] = "/dev/null/__no_file__"

@contextmanager
def quiet():
    """Suppress stdout from the simulation (machine/router/source prints)."""
    old = sys.stdout
    sys.stdout = open(os.devnull, "w")
    try:
        yield
    finally:
        sys.stdout.close()
        sys.stdout = old

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kata.core.config import KATAConfig
from kata.core.tokenizer import StateTokenizer
from kata.env import KataEnv
from kata.scenario import ScenarioBuilder
from kata.EntityFactories import RandomScenarioSampler
from experiment.config import AgentConfig
from agents import (
    LeastBusyAgent,
    LeastFatiguedAgent,
    PPOTransformerAgent,
    RainbowDQNAgent,
    RandomAgent,
    RoundRobinAgent,
    ShortestQueueAgent,
)
from agents.grpo.grpo import GRPOAgent

_AGENT_REGISTRY = {
    "random":          RandomAgent,
    "round_robin":     RoundRobinAgent,
    "least_busy":      LeastBusyAgent,
    "least_fatigued":  LeastFatiguedAgent,
    "shortest_queue":  ShortestQueueAgent,
    "rainbow_dqn":     RainbowDQNAgent,
    "grpo":            GRPOAgent,
    "ppo_transformer": PPOTransformerAgent,
}
_TOKEN_AGENTS = {"rainbow_dqn", "grpo", "ppo_transformer"}

print("Imports OK")

## 3. Build the shared evaluation scenario

We sample **one** factory from the trained agent's config pool and reuse it for every agent. The horizon is overridden so the benchmark probes long-run behaviour irrespective of the training-time `max_sim_time`. A tokenizer is built from the full type pool so the trained agent never sees unknown tokens.

In [ ]:
env_cfg = KATAConfig(**json.loads(TRAINED_ENV_CONFIG.read_text()))

# Stretch the horizon for the benchmark (does not touch the saved env config file)
env_cfg.gym = env_cfg.gym.model_copy(update={
    "max_episode_steps": MAX_EVAL_STEPS,
    "max_sim_time":      MAX_EVAL_SIM_TIME,
})

rcfg = env_cfg.randomized_scenario
if rcfg.enabled:
    sampler = RandomScenarioSampler(env_cfg, rcfg, seed=EVAL_SEED)
    fixed_eval_cfg = sampler.sample_config()
    def scenario_factory(c=fixed_eval_cfg):
        return ScenarioBuilder(c).build()
    n_techs        = rcfg.n_technicians
    machine_types  = sampler.all_machine_types()
    component_types = sampler.all_component_types()
else:
    def scenario_factory(c=env_cfg):
        return ScenarioBuilder(c).build()
    n_techs        = len(env_cfg.technicians)
    machine_types  = sorted({m.machine_type for m in env_cfg.machines.values()})
    component_types = sorted({
        c.component_type
        for m in env_cfg.machines.values()
        for c in m.components.values()
    })

tokenizer = StateTokenizer.build_vocab(
    machine_types=machine_types,
    n_technicians=n_techs,
    seq_length=env_cfg.gym.tokenizer_seq_length,
    component_types=component_types,
    next_ticket_lookahead=env_cfg.gym.next_ticket_lookahead,
)

print(f"Action space size : {n_techs}")
print(f"Machine types     : {machine_types}")
print(f"Tokenizer vocab   : {tokenizer.vocab_size}")

## 4. Instantiate the trained agent and the baselines

The trained agent's `params` are read straight from its training JSON, so the network shape matches the checkpoint. We then `load()` the weights.

Each agent gets its **own** `KataEnv` paired with the same shared `scenario_factory`. Token-based agents (the trained one) use `observation_representation="token_ids"`; heuristic baselines use `"structured"` so they have access to `technician_busy` / `technician_fatigue`.

In [ ]:
def _make_env(representation: str) -> KataEnv:
    """Build a fresh env on the shared scenario_factory, with the given obs format."""
    gym_cfg = env_cfg.gym.model_copy(update={"observation_representation": representation})
    with quiet():
        env = KataEnv(
            scenario_factory=scenario_factory,
            config=gym_cfg,
            tokenizer=tokenizer if representation == "token_ids" else None,
        )
    return env

def _peek_checkpoint_vocab_size(path: Path) -> int | None:
    """Read the vocab_size embedded in a torch checkpoint, if any.

    Lets us fail fast when the eval-time tokenizer does not match the
    one used to train the saved weights — the embedding row dimension
    encodes the training-time vocab size exactly.
    """
    import torch
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt.get("net") if isinstance(ckpt, dict) else None
    if not isinstance(state, dict):
        state = ckpt if isinstance(ckpt, dict) else {}
    for k, v in state.items():
        if "token_embedding" in k and hasattr(v, "shape") and len(v.shape) == 2:
            return int(v.shape[0])
    return None

agents: dict[str, tuple["Agent", KataEnv]] = {}

# -- trained agent ----------------------------------------------------------
agent_cfg = AgentConfig(**json.loads(TRAINED_AGENT_CONFIG.read_text()))
trained_cls = _AGENT_REGISTRY[agent_cfg.agent_type]
trained_params = dict(agent_cfg.params)
trained_params["n_actions"] = n_techs

if agent_cfg.agent_type in _TOKEN_AGENTS:
    ckpt_vocab = _peek_checkpoint_vocab_size(TRAINED_CHECKPOINT)
    if ckpt_vocab is not None and ckpt_vocab != tokenizer.vocab_size:
        raise RuntimeError(
            f"Vocab size mismatch — the checkpoint was trained with {ckpt_vocab} "
            f"tokens but the eval-time tokenizer has {tokenizer.vocab_size}.\n"
            f"This happens when TRAINED_ENV_CONFIG ({TRAINED_ENV_CONFIG}) uses "
            f"a different machine_templates pool, component pool or "
            f"n_technicians than the env that produced the checkpoint.\n"
            f"Fix: point TRAINED_ENV_CONFIG at the env JSON that was used to "
            f"train this checkpoint (you can confirm by checking "
            f"reports/<exp_id>/config.json from the training run)."
        )
    trained_params.setdefault("vocab_size", tokenizer.vocab_size)
    trained_params.setdefault("max_seq_len", env_cfg.gym.tokenizer_seq_length)

trained_repr = "token_ids" if agent_cfg.agent_type in _TOKEN_AGENTS else "structured"
trained_env = _make_env(trained_repr)
trained_agent = trained_cls(**trained_params)
trained_agent.load(TRAINED_CHECKPOINT)
agents[TRAINED_AGENT_LABEL] = (trained_agent, trained_env)
print(f"Loaded {agent_cfg.agent_type:>16s} ({TRAINED_AGENT_LABEL}) from {TRAINED_CHECKPOINT}")

# -- baselines --------------------------------------------------------------
for name in BASELINES:
    cls = _AGENT_REGISTRY[name]
    env = _make_env("structured")
    agents[name] = (cls(n_actions=n_techs), env)
    print(f"Built  {name:>16s}")

print(f"\nAgents under benchmark: {list(agents.keys())}")

## 5. Run the benchmark

Each `(agent, env)` pair runs `N_EVAL_EPISODES` rollouts. We collect:

- **Episode KPIs** (from the env's terminal `info["metrics"]`): finished products, MTTR, fleet availability, total breakdowns, total assignments…
- **Per-step series** (from `info["metrics"]`): `repair_time_delta_per`, `repair_quality`, `tech_knowledge`. Stored as lists, one per episode.
- **Episode-cumulative reward**.

In [ ]:
EPISODE_KPI_KEYS = [
    "finished_products",
    "mttr",
    "fleet_availability_rate",
    "throughput_rate",
    "total_breakdowns",
    "total_assignments",
    "total_repairs",
    "ill_technician_count",
]
STEP_SERIES_KEYS = ["repair_time_delta_per", "repair_quality"]

def _scalar_step_metrics(metrics: dict) -> dict[str, float]:
    """Drop grouped (`name/subkey`) entries and keep plain scalars."""
    return {k: float(v) for k, v in metrics.items() if "/" not in k}

def _mean_grouped(metrics: dict, prefix: str) -> float | None:
    """Mean across `prefix/<subkey>` entries (e.g. fleet-wide tech_knowledge)."""
    vals = [float(v) for k, v in metrics.items() if k.startswith(prefix + "/")]
    return float(np.mean(vals)) if vals else None

def run_episode(agent, env, *, seed: int) -> dict:
    """Run one rollout to termination and collect metrics."""
    agent.on_episode_start()
    with quiet():
        obs, _info = env.reset(seed=seed)
    ep_reward = 0.0
    step_series: dict[str, list[float]] = {k: [] for k in STEP_SERIES_KEYS}
    fleet_knowledge_series: list[float] = []
    final_info: dict = {}
    n_steps = 0
    while True:
        action = agent.select_action(obs, deterministic=DETERMINISTIC)
        with quiet():
            obs, reward, terminated, truncated, info = env.step(action)
        ep_reward += float(reward)
        n_steps += 1
        metrics = info.get("metrics", {})
        for k in STEP_SERIES_KEYS:
            if k in metrics:
                step_series[k].append(float(metrics[k]))
        fk = _mean_grouped(metrics, "tech_knowledge")
        if fk is not None:
            fleet_knowledge_series.append(fk)
        if terminated or truncated:
            final_info = info
            break
    agent.on_episode_end(ep_reward)
    ep_kpis = _scalar_step_metrics(final_info.get("metrics", {}))
    ep_kpis["episode_reward"] = ep_reward
    ep_kpis["n_steps"]        = n_steps
    ep_kpis["sim_time"]       = float(final_info.get("sim_time", 0.0))
    return {
        "kpis": ep_kpis,
        "step_series": step_series,
        "fleet_knowledge": fleet_knowledge_series,
    }

results: dict[str, list[dict]] = {label: [] for label in agents}
t0 = time.time()
for label, (agent, env) in agents.items():
    print(f"--- {label} ---")
    for ep in range(N_EVAL_EPISODES):
        seed = EVAL_SEED * 100 + ep   # deterministic per-episode seed shared across agents
        ep_result = run_episode(agent, env, seed=seed)
        results[label].append(ep_result)
        kp = ep_result["kpis"]
        print(
            f"  ep{ep}  finished={kp.get('finished_products', float('nan')):.0f}"
            f"  mttr={kp.get('mttr', float('nan')):.2f}"
            f"  avail={kp.get('fleet_availability_rate', float('nan')):.3f}"
            f"  return={kp['episode_reward']:+.2f}"
            f"  steps={kp['n_steps']}"
        )
print(f"\nTotal wall time: {time.time() - t0:.1f}s")

## 6. Per-episode KPI summary

Mean ± std across `N_EVAL_EPISODES` episodes for each agent.

In [ ]:
rows = []
for label, ep_list in results.items():
    for ep_idx, ep in enumerate(ep_list):
        row = {"agent": label, "episode": ep_idx}
        row.update(ep["kpis"])
        rows.append(row)
ep_df = pd.DataFrame(rows)

summary_cols = [c for c in (EPISODE_KPI_KEYS + ["episode_reward"]) if c in ep_df.columns]
summary = ep_df.groupby("agent")[summary_cols].agg(["mean", "std"]).round(3)
summary

## 7. Headline KPI bar charts

One panel per KPI. Bars are episode means; error bars are ±1 std.

In [ ]:
plot_kpis = [c for c in [
    "episode_reward",
    "finished_products",
    "fleet_availability_rate",
    "mttr",
    "total_breakdowns",
    "throughput_rate",
] if c in ep_df.columns]

n = len(plot_kpis)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.2 * nrows))
axes = np.atleast_2d(axes).flatten()

agent_order = list(agents.keys())
for i, kpi in enumerate(plot_kpis):
    ax = axes[i]
    means = [ep_df[ep_df.agent == a][kpi].mean() for a in agent_order]
    stds  = [ep_df[ep_df.agent == a][kpi].std(ddof=0) for a in agent_order]
    colors = ["tab:red" if a == TRAINED_AGENT_LABEL else "tab:blue" for a in agent_order]
    ax.bar(agent_order, means, yerr=stds, color=colors, edgecolor="black", linewidth=0.5, capsize=4)
    ax.set_title(kpi)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.3)
for j in range(n, len(axes)):
    axes[j].axis("off")
fig.suptitle(f"Agent benchmark ({N_EVAL_EPISODES} eps × max_sim_time={MAX_EVAL_SIM_TIME:.0f})", y=1.02)
fig.tight_layout()
plt.show()

## 8. Per-step trend comparison

Within-episode time-series, averaged across the `N_EVAL_EPISODES` rollouts and smoothed with a rolling mean. Helps see how policies behave *during* a long episode (e.g. knowledge growth, repair-time savings drifting).

In [ ]:
def _stack_series(series_per_episode: list[list[float]]) -> np.ndarray:
    """Pad ragged episodes with NaN to a 2-D array of shape (n_eps, max_len)."""
    if not series_per_episode:
        return np.zeros((0, 0))
    max_len = max(len(s) for s in series_per_episode)
    out = np.full((len(series_per_episode), max_len), np.nan)
    for i, s in enumerate(series_per_episode):
        out[i, :len(s)] = s
    return out

def _rolling_mean(x: np.ndarray, w: int) -> np.ndarray:
    w = max(1, min(int(w), len(x)))
    if w <= 1:
        return x
    out = np.empty_like(x, dtype=float)
    csum = np.concatenate([[0.0], np.nancumsum(x)])
    counts = np.concatenate([[0], np.cumsum(~np.isnan(x))])
    for i in range(len(x)):
        lo = max(0, i + 1 - w)
        denom = max(1, counts[i + 1] - counts[lo])
        out[i] = (csum[i + 1] - csum[lo]) / denom
    return out

trend_metrics = STEP_SERIES_KEYS + ["fleet_knowledge"]
fig, axes = plt.subplots(len(trend_metrics), 1, figsize=(9, 3.2 * len(trend_metrics)))
axes = np.atleast_1d(axes)

for ax, metric in zip(axes, trend_metrics):
    for label in agent_order:
        if metric == "fleet_knowledge":
            per_ep = [ep["fleet_knowledge"] for ep in results[label]]
        else:
            per_ep = [ep["step_series"].get(metric, []) for ep in results[label]]
        stacked = _stack_series(per_ep)
        if stacked.size == 0:
            continue
        mean_curve = np.nanmean(stacked, axis=0)
        w = max(5, len(mean_curve) // 30)
        smoothed = _rolling_mean(mean_curve, w)
        is_trained = (label == TRAINED_AGENT_LABEL)
        ax.plot(
            range(len(smoothed)),
            smoothed,
            linewidth=2.0 if is_trained else 1.0,
            alpha=1.0 if is_trained else 0.7,
            color="red" if is_trained else None,
            label=label,
        )
    ax.set_title(metric)
    ax.set_xlabel("step within episode")
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
fig.tight_layout()
plt.show()

## 9. Persist the results

Drops the per-episode KPI table to `reports/benchmark_<timestamp>/episode_metrics.csv` and the per-step series of each agent's first episode (most useful for plot reproduction).

In [ ]:
ts = time.strftime("%Y%m%d-%H%M%S")
out_dir = REPORTS_ROOT / f"benchmark_{ts}"
out_dir.mkdir(parents=True, exist_ok=True)

ep_df.to_csv(out_dir / "episode_metrics.csv", index=False)
summary.to_csv(out_dir / "kpi_summary.csv")

for label, ep_list in results.items():
    if not ep_list:
        continue
    safe_label = label.replace("/", "_").replace(" ", "_")
    first = ep_list[0]
    rows_step = []
    max_len = max(
        [len(s) for s in first["step_series"].values()]
        + [len(first["fleet_knowledge"])]
        + [0]
    )
    for i in range(max_len):
        row = {"step": i}
        for k, series in first["step_series"].items():
            row[k] = series[i] if i < len(series) else None
        row["fleet_knowledge"] = (
            first["fleet_knowledge"][i] if i < len(first["fleet_knowledge"]) else None
        )
        rows_step.append(row)
    pd.DataFrame(rows_step).to_csv(out_dir / f"step_series__{safe_label}.csv", index=False)

manifest = {
    "trained_env_config":   str(TRAINED_ENV_CONFIG),
    "trained_agent_config": str(TRAINED_AGENT_CONFIG),
    "trained_checkpoint":   str(TRAINED_CHECKPOINT),
    "trained_agent_label":  TRAINED_AGENT_LABEL,
    "baselines":            BASELINES,
    "n_eval_episodes":      N_EVAL_EPISODES,
    "max_eval_sim_time":    MAX_EVAL_SIM_TIME,
    "max_eval_steps":       MAX_EVAL_STEPS,
    "eval_seed":            EVAL_SEED,
    "deterministic":        DETERMINISTIC,
    "n_techs":              n_techs,
    "machine_types":        machine_types,
    "component_types":      component_types,
    "tokenizer_vocab_size": tokenizer.vocab_size,
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))

print(f"Saved benchmark artefacts to {out_dir}")
for p in sorted(out_dir.iterdir()):
    print("  ", p.name)